# Timestral Forecasting Pipeline
# =============================

A self-contained Jupyter notebook demonstrating time series forecasting using Mistral AI's Timestral foundation model.
Uses the classic AirPassengers dataset for zero-shot forecasting.

Requirements: pandas, numpy, matplotlib, torch, timestral

## 1. Setup & Installation

Timestral is Mistral AI's family of time series foundation models. We install it directly from GitHub.

In [ ]:
# Install timestral from GitHub
# Note: This may take a few minutes the first time
!pip install git+https://github.com/mistralai/timestral.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Installation complete. Package versions:")
print(f"torch: {torch.__version__}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")

## 2. Data Loading

We'll use the same AirPassengers dataset as the ARIMA notebook for comparison.

In [ ]:
# Load the classic AirPassengers dataset
from statsmodels.datasets import get_rdataset
data = get_rdataset('AirPassengers', 'datasets')
df = data.data

# Convert to time series with datetime index
df['time'] = pd.to_datetime(df['time'])
df.set_index('time', inplace=True)
df = df[['value']]
df.columns = ['passengers']

print(df.head())

## 3. Exploratory Data Analysis

In [ ]:
# Plot the time series
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['passengers'], label='Passengers')
plt.title('AirPassengers Time Series')
plt.xlabel('Date')
plt.ylabel('Number of Passengers')
plt.grid(True)
plt.legend()
plt.show()

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Missing values: {df.isnull().sum().values}")

## 4. Data Preparation for Timestral

Timestral expects numpy arrays or torch tensors. We'll prepare the data accordingly.

Key considerations:
- Timestral is a zero-shot model, so no training is required
- The model expects input shape (T,) for univariate series
- We need to handle the datetime index appropriately

In [ ]:
from timestral import TimestralForecaster

# Convert to numpy array
series_array = df['passengers'].values

# Split into train and test (last 12 months for testing)
train_array = series_array[:-12]
test_array = series_array[-12:]

print(f"Train shape: {train_array.shape}")
print(f"Test shape: {test_array.shape}")
print(f"Train mean: {train_array.mean():.2f}, std: {train_array.std():.2f}")
print(f"Test mean: {test_array.mean():.2f}, std: {test_array.std():.2f}")

## 5. Load Timestral Model

We'll use the Timestral-209M model which is the largest and most accurate in the family.
Available models: Timestral-5M, Timestral-28M, Timestral-123M, Timestral-209M

Note: The first download may take several minutes depending on your internet connection.

In [ ]:
# Initialize the Timestral forecaster
# Using 'mistralai/Timestral-209M' model
# device='cpu' uses CPU, use 'cuda' if you have a GPU
model = TimestralForecaster.from_pretrained('mistralai/Timestral-209M', device='cpu')

print(f"Model loaded: {model}")

## 6. Forecasting with Timestral

Timestral performs zero-shot forecasting, meaning it can forecast on unseen data without training.
We'll forecast for the test period (12 months ahead).

In [ ]:
# Convert train data to torch tensor
train_tensor = torch.from_numpy(train_array).float()

# Forecast 12 steps ahead (same as test size)
forecast_steps = len(test_array)
forecast = model.forecast(train_tensor.unsqueeze(0), horizon=forecast_steps)

# Extract forecast values
forecast_values = forecast.mean.squeeze().numpy()

print(f"Forecast shape: {forecast_values.shape}")
print(f"Forecast values: {forecast_values}")

## 7. Visualization

In [ ]:
# Create date range for forecast
last_train_date = df.index[-13]  # -13 because we kept last 12 for test
forecast_dates = pd.date_range(start=last_train_date + pd.DateOffset(months=1), periods=forecast_steps, freq='MS')

# Plot results
plt.figure(figsize=(14, 7))
plt.plot(df.index[:-12], train_array, label='Training Data', color='blue')
plt.plot(df.index[-12:], test_array, label='Actual Test Data', color='green', linewidth=2)
plt.plot(forecast_dates, forecast_values, label='Timestral Forecast', color='red', linestyle='--', linewidth=2)
plt.title('Timestral Zero-Shot Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.legend()
plt.grid(True)
plt.show()

# Plot with confidence intervals if available
if hasattr(forecast, 'std'):
    forecast_std = forecast.std.squeeze().numpy()
    upper = forecast_values + 1.96 * forecast_std
    lower = forecast_values - 1.96 * forecast_std
    
    plt.figure(figsize=(14, 7))
    plt.plot(df.index[:-12], train_array, label='Training Data', color='blue')
    plt.plot(df.index[-12:], test_array, label='Actual Test Data', color='green', linewidth=2)
    plt.plot(forecast_dates, forecast_values, label='Timestral Forecast', color='red', linestyle='--', linewidth=2)
    plt.fill_between(forecast_dates, lower, upper, color='pink', alpha=0.3, label='95% Confidence Interval')
    plt.title('Timestral Forecast with Confidence Intervals')
    plt.xlabel('Date')
    plt.ylabel('Passengers')
    plt.legend()
    plt.grid(True)
    plt.show()

## 8. Evaluation

In [ ]:
# Calculate metrics
mae = mean_absolute_error(test_array, forecast_values)
rmse = np.sqrt(mean_squared_error(test_array, forecast_values))
mape = np.mean(np.abs((test_array - forecast_values) / test_array)) * 100

print("Timestral Model Performance:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

# Plot residuals
residuals = test_array - forecast_values
plt.figure(figsize=(12, 6))
plt.plot(forecast_dates, residuals, marker='o', linestyle='-')
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Residuals of Timestral Forecast')
plt.xlabel('Date')
plt.ylabel('Residuals')
plt.grid(True)
plt.show()

# Summary statistics of residuals
print(f"\nResiduals mean: {residuals.mean():.2f}")
print(f"Residuals std: {residuals.std():.2f}")

## 9. Comparison with ARIMA

Let's compare Timestral's performance with the ARIMA baseline from the other notebook.

In [ ]:
# Fit ARIMA model for comparison
from statsmodels.tsa.arima.model import ARIMA

# Use the same train/test split
train_series = df.iloc[:-12]['passengers']
test_series = df.iloc[-12:]['passengers']

# Fit ARIMA(1,1,1) with seasonal component
arima_model = ARIMA(train_series, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12))
arima_fit = arima_model.fit()

# Forecast with ARIMA
arima_forecast = arima_fit.get_forecast(steps=forecast_steps)
arima_forecast_values = arima_forecast.predicted_mean.values

# Calculate ARIMA metrics
arima_mae = mean_absolute_error(test_series, arima_forecast_values)
arima_rmse = np.sqrt(mean_squared_error(test_series, arima_forecast_values))
arima_mape = np.mean(np.abs((test_series - arima_forecast_values) / test_series)) * 100

# Print comparison
print("\n=== Performance Comparison ===\n")
print(f"{'Model':<20} {'MAE':<15} {'RMSE':<15} {'MAPE':<15}")
print(f"{'Timestral':<20} {mae:<15.2f} {rmse:<15.2f} {mape:<15.2f}")
print(f"{'ARIMA':<20} {arima_mae:<15.2f} {arima_rmse:<15.2f} {arima_mape:<15.2f}")

# Plot both forecasts
plt.figure(figsize=(14, 7))
plt.plot(df.index[:-12], train_series, label='Training Data', color='blue')
plt.plot(df.index[-12:], test_series, label='Actual Test Data', color='green', linewidth=2)
plt.plot(forecast_dates, forecast_values, label='Timestral Forecast', color='red', linestyle='--', linewidth=2)
plt.plot(forecast_dates, arima_forecast_values, label='ARIMA Forecast', color='purple', linestyle=':', linewidth=2)
plt.title('Timestral vs ARIMA Forecast Comparison')
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.legend()
plt.grid(True)
plt.show()

## 10. Advanced Usage

Timestral supports multivariate forecasting and covariates. Here's how to use it with multiple series.

In [ ]:
# Example: Multivariate forecasting (if we had multiple series)
# This is just an example - our dataset is univariate
# For multivariate, you would have shape (B, V, T) where:
# - B: batch size
# - V: number of variates (series)
# - T: time steps

print("Multivariate example:")
print("# Create multiple series (example with dummy data)")
print("multivariate_data = torch.randn(2, 100)  # 2 series, 100 time steps")
print("# forecast = model.forecast(multivariate_data.unsqueeze(0), horizon=12)")
print("\nNote: Timestral can handle:")
print("- Univariate time series (single series)")
print("- Multivariate time series (multiple related series)")
print("- Past-only covariates (features that don't depend on future)")
print("- Known-future covariates (features available for forecast horizon)")

## 11. Save Forecast Results (Optional)

In [ ]:
# Create a DataFrame with forecast results
forecast_df = pd.DataFrame({
    'date': forecast_dates,
    'actual': test_array,
    'forecast': forecast_values,
    'residual': residuals
})

# Save to CSV
forecast_df.to_csv('timestral_forecast_results.csv', index=False)
print("Forecast results saved to timestral_forecast_results.csv")
print(forecast_df.head())

## 12. Cleanup (Optional)

In [ ]:
# Clear GPU cache if using CUDA
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared")

# Delete model to free memory
del model
print("Model deleted from memory")

## Key Takeaways

### What we've demonstrated:
1. **Zero-shot forecasting**: Timestral can forecast on unseen data without training
2. **Easy integration**: Simple API similar to scikit-learn's fit/predict pattern
3. **High accuracy**: Foundation models trained on vast amounts of time series data
4. **Flexible input**: Supports univariate and multivariate series

### When to use Timestral:
- When you need quick forecasts without model training
- When you have limited historical data
- When you need to forecast many different series
- When traditional models (ARIMA, Prophet) are not accurate enough

### Limitations:
- Requires significant memory (especially larger models)
- First inference may be slower due to model loading
- Less interpretable than traditional statistical models

### Next Steps:
- Try different Timestral model sizes (5M, 28M, 123M, 209M)
- Experiment with multivariate forecasting
- Add exogenous variables (covariates)
- Implement proper backtesting and cross-validation
- Deploy as a forecasting service

## References

- [Timestral GitHub](https://github.com/mistralai/timestral)
- [Mistral AI Blog](https://mistral.ai/news/)
- [Timestral Model Cards on Hugging Face](https://huggingface.co/collections/mistralai/timestral-66c558673969b32284353659)